**Load Silver Table**

In [0]:
silver_df = spark.table("ecommerce.silver.incremental_orders_silver")
display(silver_df)

order_id,order_date,customer_name,product,category,quantity,price,ingested_at,discount_percentage
2001,2026-09-05,Ravi,Laptop,Electronics,1,800.0,2026-10-09T16:50:19.501Z,null
2002,2026-09-06,John,Mouse,Electronics,2,25.0,2026-10-09T16:50:19.501Z,null
2003,2026-09-06,Priya,Shoes,Fashion,1,60.0,2026-10-09T16:50:19.501Z,null
2004,2026-09-05,David,Keyboard,Electronics,1,45.0,2026-10-09T16:50:19.501Z,null
2005,2026-09-05,Anitha,T-Shirt,Fashion,2,20.0,2026-10-09T16:50:19.501Z,null
2006,2026-09-05,Robert,Headphones,Electronics,1,75.0,2026-10-09T16:50:19.501Z,null
2007,2026-09-05,Suresh,Jeans,Fashion,1,40.0,2026-10-09T16:50:19.501Z,null
2008,2026-09-05,Emily,Coffee Maker,Home,1,90.0,2026-10-09T16:50:19.501Z,null
2009,2026-09-05,Arun,Monitor,Electronics,1,250.0,2026-10-09T16:50:19.501Z,null
2010,2026-09-05,Sarah,Backpack,Fashion,1,35.0,2026-10-09T16:50:19.501Z,null


**Daily Revenue (Gold Table 1)**

In [0]:
from pyspark.sql.functions import col, sum as _sum

daily_revenue_df = (
    silver_df
    .groupBy("order_date")
    .agg(
        _sum(col("quantity") * col("price")).alias("total_revenue")
    )
    .orderBy("order_date")
)

display(daily_revenue_df)


order_date,total_revenue
2026-09-05,1375.0
2026-09-06,1730.0
2026-09-07,2140.0


**Create Gold table**

In [0]:
daily_revenue_df.write.mode("overwrite").saveAsTable("ecommerce.gold.daily_revenue")


**Category Revenue (Gold Table 2)**

In [0]:
category_revenue_df = (
    silver_df
    .groupBy("category")
    .agg(
        _sum(col("quantity") * col("price")).alias("category_revenue")
    )
    .orderBy("category")
)

display(category_revenue_df)


category,category_revenue
Electronics,4160.0
Fashion,585.0
Home,500.0


Create Gold table 2

In [0]:
category_revenue_df.write.mode("overwrite").saveAsTable("ecommerce.gold.category_revenue")


**Top Customers (Gold Table 3)**

In [0]:
top_customers_df = (
    silver_df
    .groupBy("customer_name")
    .agg(
        _sum(col("quantity") * col("price")).alias("total_spent")
    )
    .orderBy(col("total_spent").desc())
)

display(top_customers_df)


customer_name,total_spent
Ravi,1495.0
John,970.0
Priya,900.0
Sarah,465.0
Arun,415.0
Suresh,260.0
Robert,220.0
David,185.0
Emily,180.0
Anitha,155.0


**Create Gold table**

In [0]:
top_customers_df.write.mode("overwrite").saveAsTable("ecommerce.gold.top_customers")


**Validate Gold Tables**

In [0]:
display(spark.table("ecommerce.gold.daily_revenue"))
display(spark.table("ecommerce.gold.category_revenue"))
display(spark.table("ecommerce.gold.top_customers"))


order_date,total_revenue
2026-09-05,1375.0
2026-09-06,1730.0
2026-09-07,2140.0


category,category_revenue
Electronics,4160.0
Fashion,585.0
Home,500.0


customer_name,total_spent
Ravi,1495.0
John,970.0
Priya,900.0
Sarah,465.0
Arun,415.0
Suresh,260.0
Robert,220.0
David,185.0
Emily,180.0
Anitha,155.0


In [0]:
display(spark.table("ecommerce.gold.top_customers"))

customer_name,total_spent
Ravi,1495.0
John,970.0
Priya,900.0
Sarah,465.0
Arun,415.0
Suresh,260.0
Robert,220.0
David,185.0
Emily,180.0
Anitha,155.0
